# 3c – Volle Ausbaustufe: Hybrid Search + Reranking + Relevanz-Gate

Aufbauend auf Notebook 3b. Der Graph bleibt unverändert – neu ist, **womit** gesucht wird:
statt nur der Vektordatenbank kommt jetzt ein **Ensemble aus Vektorsuche und BM25** zum Einsatz.

Damit steht die dreistufige Architektur, die moderne RAG-Systeme üblicherweise verwenden:

```
breit suchen          intelligent filtern        antworten
(Vektor + BM25)   →   (Cross-Encoder)        →   (LLM)
```

> **Voraussetzung:** Notebook 1 muss mit `BUILD_BM25_INDEX = True` gelaufen sein –
> sonst fehlt `bm25_index/chunks.jsonl`.

> **Wichtig:** Beide Indizes müssen aus **demselben Lauf** von Notebook 1 stammen. Nicht nur
> derselbe Dokumenten-Corpus, sondern dieselben Chunks: sonst verschmilzt die Fusion
> Textausschnitte unterschiedlichen Zuschnitts und die Quellenangaben zeigen ins Leere.

## Benötigte Pakete

```bash
pip install -U langchain-core langchain-community langchain-openai langchain-chroma \
               langgraph gradio sentence-transformers torch rank-bm25
# nur bei EMBEDDING_BACKEND = "huggingface":
pip install -U "langchain-huggingface[full]"
```

## Konfiguration

Wortgleich mit Notebook 1 und 2. Abschnitte, die dieses Notebook nicht braucht, stören hier nicht.

In [ ]:
# ============================================================
#  KONFIGURATION – in allen Notebooks der Reihe identisch
# ============================================================

# --- 1) Embedding-Backend -----------------------------------
#     Indexierung und Retrieval MÜSSEN dasselbe Backend nutzen.
EMBEDDING_BACKEND = "lmstudio"        # "lmstudio" | "huggingface"

# --- 2) LLM-Backend (ab Notebook 2) --------------------------
LLM_BACKEND = "lmstudio"              # "lmstudio" | "deepinfra" | "openrouter"

# --- 3) API-Keys der Cloud-Anbieter --------------------------
#     Nur ausfüllen, wenn das jeweilige Backend genutzt wird.
#     Bitte den Key für dich behalten: nicht weitergeben, nicht committen,
#     nicht in geteilten Notebooks stehen lassen.
DEEPINFRA_API_KEY  = ""
OPENROUTER_API_KEY = ""

# --- 4) Pfade ------------------------------------------------
DOC_SOURCE_DIR = "./documents"        # Quell-Dokumente (PDF + DOCX)
DB_DIR         = "./chroma_db"        # Vektordatenbank
BM25_DIR       = "./bm25_index"       # Lexikalischer Index
MODEL_PATH     = "./models"           # Modell-Cache (nur HuggingFace)
MANIFEST_PATH  = "./index_manifest.json"

COLLECTION_NAME = "langchain"         # muss in allen Notebooks gleich sein

# --- 5) Chunking (Notebook 1) --------------------------------
#     Ändert man das hier, muss der komplette Index neu gebaut werden.
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128                   # ~25 % Überlappung – gut für lange deutsche Sätze

# --- 6) Welche Indizes bauen? (Notebook 1) -------------------
BUILD_VECTOR_INDEX = True             # semantisch, braucht das Embedding-Modell
BUILD_BM25_INDEX   = True             # lexikalisch, braucht kein Modell

# --- 7) Einfaches Retrieval (Notebook 2) ---------------------
TOP_K = 10                            # Anzahl der Chunks pro Frage

# --- 8) Reranking & Relevanz-Gate (ab Notebook 3) ------------
RERANKER_MODEL      = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
RETRIEVE_K          = 15              # Kandidaten aus der Suche (breit suchen)
RERANK_TOP_N        = 5               # davon die N besten behalten
RELEVANCE_THRESHOLD = 0.3             # Mindest-Score des besten Chunks (0…1)
MAX_RETRIES         = 2               # max. Query-Reformulierungen

# --- 9) Hybrid Search (Notebook 3c) --------------------------
RETRIEVER_K      = 10                 # Kandidaten je Retriever vor der Fusion
ENSEMBLE_WEIGHTS = [0.5, 0.5]         # [Vektor, BM25] – Summe 1.0

# ============================================================
#  Backend-Details – normalerweise unverändert lassen
# ============================================================

LM_STUDIO_URL = "http://localhost:1234/v1"

EMBEDDING_MODELS = {
    "huggingface": "intfloat/multilingual-e5-large-instruct",
    "lmstudio":    "text-embedding-multilingual-e5-large-instruct",
}

LLM_CONFIG = {
    "lmstudio": {
        "base_url": LM_STUDIO_URL,
        "model":    "qwen/qwen3.5-9b",
        "api_key":  "lm-studio",                  # LM Studio prüft den Key nicht
    },
    "deepinfra": {
        "base_url": "https://api.deepinfra.com/v1/openai",
        "model":    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "api_key":  DEEPINFRA_API_KEY,
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "model":    "meta-llama/llama-3.3-70b-instruct",
        "api_key":  OPENROUTER_API_KEY,
    },
}

print(f"Embeddings: {EMBEDDING_BACKEND}  |  LLM: {LLM_BACKEND}  |  Gewichte: {ENSEMBLE_WEIGHTS}")

## Imports

In [ ]:
import os
import json
import base64
from typing import List, TypedDict
from urllib.parse import quote

import torch
import gradio as gr
from IPython import display

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

# Cross-Encoder für das Reranking
from sentence_transformers import CrossEncoder

# Hybrid Search: BM25 liegt in langchain_community,
# der EnsembleRetriever seit LangChain 1.0 in langchain_classic
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

## Womit wurde der Index gebaut?

Notebook 1 hinterlegt ein Manifest. Diese Zelle **gibt es nur aus** – sie prüft nichts.

> ⚠️ Vergleiche `embedding_backend` und `embedding_modell` mit deiner Konfiguration oben.

In [ ]:
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, encoding="utf-8") as f:
        manifest = json.load(f)

    print(f"📝 Index-Manifest ({MANIFEST_PATH}):")
    for key, value in manifest.items():
        print(f"   {key:<18} {value}")
else:
    print(f"ℹ️  Kein Manifest unter '{MANIFEST_PATH}' gefunden.")
    print("   Der Index stammt vermutlich aus einem älteren Lauf von Notebook 1.")

## Embedding-Modell & Vektordatenbank

Identisch zu Notebook 1 und 2: E5 erwartet die Präfixe `passage:` beim Indexieren und
`query:` beim Suchen, die der Wrapper automatisch setzt.

In [ ]:
class E5OpenAIEmbeddings(OpenAIEmbeddings):
    """E5-Präfixe für OpenAI-kompatible Endpunkte (LM Studio)."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


def build_embeddings():
    """Erzeugt das Embedding-Modell passend zu EMBEDDING_BACKEND."""
    if EMBEDDING_BACKEND == "lmstudio":
        return E5OpenAIEmbeddings(
            model=EMBEDDING_MODELS["lmstudio"],
            api_key="lm-studio",
            base_url=LM_STUDIO_URL,
            check_embedding_ctx_length=False,
        )

    if EMBEDDING_BACKEND == "huggingface":
        # Import erst hier, damit LM-Studio-Nutzer das Paket nicht brauchen
        from langchain_huggingface import HuggingFaceEmbeddings

        class E5HuggingFaceEmbeddings(HuggingFaceEmbeddings):
            """E5-Präfixe für lokal geladene sentence-transformers-Modelle."""

            def embed_documents(self, texts: list[str]) -> list[list[float]]:
                return super().embed_documents(["passage: " + t for t in texts])

            def embed_query(self, text: str) -> list[float]:
                return super().embed_query("query: " + text)

        return E5HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODELS["huggingface"],
            cache_folder=MODEL_PATH,
        )

    raise ValueError(f"Unbekanntes EMBEDDING_BACKEND: {EMBEDDING_BACKEND}")

embeddings = build_embeddings()

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
    embedding_function=embeddings,
)

print(f"✅ Vektordatenbank geladen – {vectorstore._collection.count()} Chunks verfügbar.")

## Hybrid Search: BM25-Index laden & Ensemble bauen

Notebook 1 hat neben der Vektordatenbank einen zweiten Index geschrieben: die Chunks als
**JSONL**. Daraus baut der `BM25Retriever` seinen lexikalischen Index beim Laden in
Sekundenbruchteilen neu auf.

Der `EnsembleRetriever` verschmilzt beide Ergebnislisten per **Reciprocal Rank Fusion**:

$$\text{RRF}(d) = \sum_{r \in R} \frac{w_r}{k + \text{rank}_r(d)}$$

$k$ ist ein Glättungsparameter (Standard 60), $\text{rank}_r(d)$ die Position des Dokuments
in der Liste des jeweiligen Retrievers. Entscheidend: RRF rechnet mit **Rängen, nicht mit
Scores** – deshalb ist es egal, dass Kosinus-Ähnlichkeit und BM25-Gewicht völlig
unterschiedliche Skalen haben.

### Gewichtung

| `ENSEMBLE_WEIGHTS` | Effekt |
|---|---|
| `[0.5, 0.5]` | beide gleich – guter Startpunkt |
| `[0.3, 0.7]` | mehr BM25 – gut bei Fachbegriffen, Paragraphen, Abkürzungen |
| `[0.7, 0.3]` | mehr Vektor – gut bei umgangssprachlichen Fragen |

> **Achtung:** Der Ensemble liefert die *Vereinigung* beider Listen, also bis zu
> 2 × `RETRIEVER_K` Kandidaten. Die bewertet anschließend alle der Cross-Encoder – bei
> mehreren Reformulierungsrunden summiert sich das auf der CPU spürbar.

In [ ]:
# --- BM25-INDEX LADEN (JSONL aus Notebook 1) ---

bm25_path = os.path.join(BM25_DIR, "chunks.jsonl")

bm25_chunks = []
with open(bm25_path, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        bm25_chunks.append(
            Document(page_content=record["page_content"], metadata=record["metadata"])
        )

print(f"✅ BM25-Chunks geladen – {len(bm25_chunks)} Stück aus '{bm25_path}'")

In [ ]:
# --- ENSEMBLE-RETRIEVER AUFBAUEN ---

# 1) Vektor-Retriever (semantische Suche)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})

# 2) BM25-Retriever (lexikalische Suche) – Index wird hier zur Laufzeit gebaut
bm25_retriever = BM25Retriever.from_documents(bm25_chunks, k=RETRIEVER_K)

# 3) Ensemble: verschmilzt beide via Reciprocal Rank Fusion
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=ENSEMBLE_WEIGHTS,
)

print(f"✅ Ensemble bereit – Vektor: {ENSEMBLE_WEIGHTS[0]}, BM25: {ENSEMBLE_WEIGHTS[1]}")
print(f"   Je Retriever bis zu {RETRIEVER_K} Kandidaten, danach Fusion.")

## Reranking-Modell laden

Ein **Bi-Encoder** (unser E5-Modell) ist schnell, weil Query und Dokument getrennt
eingebettet werden. Ein **Cross-Encoder** ist langsamer, aber genauer: er bewertet Query
und Dokument *gemeinsam*.

Strategie: erst breit suchen, dann mit dem Cross-Encoder die besten Kandidaten auswählen.

> **Wichtig:** Unsere Dokumente sind auf Deutsch, wir brauchen also einen **multilingualen**
> Cross-Encoder. Ein rein englischer Reranker (z.B. `ms-marco-MiniLM`) liefert bei deutschen
> Texten systematisch zu niedrige Scores.

> **Ebenso wichtig:** MS-MARCO-Cross-Encoder geben ohne `activation_fn` **rohe Logits**
> zurück – Werte etwa zwischen −10 und +10. Die Schwelle `RELEVANCE_THRESHOLD = 0.3` wäre
> darauf angewandt sinnlos. Mit der Sigmoid liegen die Scores in [0, 1] und lassen sich als
> Relevanz-Wahrscheinlichkeit lesen.

In [ ]:
# activation_fn=Sigmoid ist hier entscheidend: ohne sie liefern MS-MARCO-
# Cross-Encoder rohe Logits (etwa -10 … +10). Mit Sigmoid sind die Scores
# Wahrscheinlichkeiten in [0, 1] – und RELEVANCE_THRESHOLD wird interpretierbar.
reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512,
    activation_fn=torch.nn.Sigmoid(),
)

print(f"✅ Reranker geladen: {RERANKER_MODEL}")

## LLM konfigurieren

In [ ]:
if LLM_BACKEND not in LLM_CONFIG:
    raise ValueError(f"Unbekanntes LLM_BACKEND: {LLM_BACKEND}")

cfg = LLM_CONFIG[LLM_BACKEND]

if not cfg["api_key"]:
    raise ValueError(f"Kein API-Key für '{LLM_BACKEND}' – bitte in der Konfigurationszelle eintragen.")

llm = ChatOpenAI(
    model=cfg["model"],
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
    max_tokens=5000,
    temperature=0,          # Für RAG: keine Kreativität, sondern Fakten
)

print(f"✅ LLM bereit: {cfg['model']} ({LLM_BACKEND}).")

## State – jetzt mit Reducer

Neu gegenüber 3a ist `query_history`. Es soll **alle** bisherigen Suchanfragen sammeln –
und genau hier reicht ein normales Feld nicht mehr aus.

In LangGraph **ersetzt** eine Node den Wert eines Feldes, wenn sie ihn zurückgibt. Der
Zyklus würde die Historie also bei jedem Durchlauf überschreiben statt sie zu verlängern.
Ein `Annotated[..., operator.add]` hängt den Rückgabewert stattdessen an das Bestehende an:

```python
query_history: Annotated[List[str], operator.add]
```

Die Node gibt `{"query_history": [neue_query]}` zurück – eine Liste mit einem Element –
und der **Reducer** `operator.add` verkettet sie mit der bisherigen. Kein `append()`, kein
Zugriff auf den alten Zustand, keine Race Conditions bei parallelen Knoten.

Der praktische Gewinn: die Reformulierung sieht jetzt alle bisherigen Fehlversuche und kann
sich davon absetzen, statt bei `temperature=0` dieselbe Umformulierung erneut zu erzeugen.

In [ ]:
import operator
from typing import Annotated


class GraphState(TypedDict):
    question:      str          # Frage des Menschen – bleibt unverändert
    search_query:  str          # Anfrage an die Suche – wird reformuliert
    context:       List[str]
    metadata:      List[dict]
    rerank_scores: List[float]
    answer:        str
    token_usage:   dict
    retry_count:   int
    query_history: Annotated[List[str], operator.add]   # ← Reducer statt Ersetzen

## Nodes – die Bausteine des Graphen

Nur `retrieve` ändert sich gegenüber 3b: es fragt den Ensemble-Retriever statt die
Vektordatenbank. Alles andere bleibt gleich – ein Hinweis darauf, wie gut sich Retrieval
und Graph-Logik voneinander trennen lassen.

In [ ]:
# --- NODE: RETRIEVE (HYBRID SEARCH) ---

def retrieve(state: GraphState) -> dict:
    """Holt Kandidaten über den Ensemble-Retriever (Vektor + BM25).

    Der EnsembleRetriever führt beide Suchen aus und verschmilzt die Ergebnisse
    via Reciprocal Rank Fusion. Duplikate werden dabei entfernt – ein Chunk, den
    beide Retriever finden, erhält einen höheren RRF-Score.
    """
    query = state.get("search_query") or state["question"]

    print(f"--- RETRIEVE / HYBRID SEARCH (Versuch {state.get('retry_count', 0) + 1}) ---")
    print(f"    Suchanfrage: {query}")

    docs = ensemble_retriever.invoke(query)

    context  = []
    metadata = []

    for i, doc in enumerate(docs):
        context.append(doc.page_content)
        source_file = os.path.basename(doc.metadata.get("source", "Unbekannt"))
        page_num    = doc.metadata.get("page", 0) + 1
        metadata.append({"id": i + 1, "source": source_file, "page": page_num})

    print(f"    → {len(docs)} Kandidaten nach Fusion (Duplikate entfernt)")
    return {"context": context, "metadata": metadata}

In [ ]:
# --- NODE: RERANK ---

def rerank(state: GraphState) -> dict:
    """Bewertet die Kandidaten mit einem Cross-Encoder und behält die Top-N."""
    print("--- RERANK ---")

    # Bewertet wird gegen die Suchanfrage, mit der die Kandidaten gefunden wurden
    query = state.get("search_query") or state["question"]

    # Cross-Encoder erwartet Paare aus (Query, Passage)
    pairs  = [(query, chunk) for chunk in state["context"]]
    scores = reranker.predict(pairs)

    # Nach Score sortieren (absteigend) und Top-N behalten
    scored_items = sorted(
        zip(scores, state["context"], state["metadata"]),
        key=lambda x: x[0],
        reverse=True,
    )
    top_items = scored_items[:RERANK_TOP_N]

    # IDs neu vergeben (1-basiert)
    reranked_context  = []
    reranked_metadata = []
    reranked_scores   = []

    for new_id, (score, text, meta) in enumerate(top_items, start=1):
        reranked_context.append(text)
        reranked_metadata.append({**meta, "id": new_id})
        reranked_scores.append(float(score))

    print(f"    → Top-{RERANK_TOP_N} Scores: {[f'{s:.3f}' for s in reranked_scores]}")

    return {
        "context":       reranked_context,
        "metadata":      reranked_metadata,
        "rerank_scores": reranked_scores,
    }

In [ ]:
# --- NODE: GENERATE ---

ANSWER_PROMPT = ChatPromptTemplate.from_template("""\
Du bist ein präziser Assistent. Beantworte die Frage NUR basierend auf dem KONTEXT.

REGELN:
1. Verweise im Text deiner Antwort auf die Abschnitte, z.B. [1] oder [Quelle: Datei.pdf, S. 5].
2. Wenn die Info nicht im Kontext ist, sag es offen.
3. Erfinde KEINE Fakten.

KONTEXT:
{context}

FRAGE: {question}
""")


def generate(state: GraphState) -> dict:
    """Erzeugt eine Antwort auf Basis der besten Kontext-Chunks."""
    print("--- GENERATE ---")

    formatted_context = ""
    for i, text in enumerate(state["context"]):
        meta  = state["metadata"][i]
        score = state["rerank_scores"][i]
        formatted_context += (
            f"\n--- ABSCHNITT {meta['id']} "
            f"(Quelle: {meta['source']}, Seite {meta['page']}, "
            f"Relevanz: {score:.3f}) ---\n"
            f"{text}\n"
        )

    # Beantwortet wird IMMER die Frage des Menschen – nie die Suchanfrage
    chain    = ANSWER_PROMPT | llm
    response = chain.invoke({"context": formatted_context, "question": state["question"]})
    usage    = response.response_metadata.get("token_usage", {})

    return {"answer": response.content, "token_usage": usage}

In [ ]:
# --- NODE: REFORMULATE ---

REFORMULATE_PROMPT = ChatPromptTemplate.from_template("""\
Die bisherigen Suchanfragen haben keine ausreichend relevanten Ergebnisse geliefert.
Formuliere eine NEUE Suchanfrage, die sich von allen bisherigen deutlich unterscheidet –
nutze Synonyme, Fachbegriffe oder einen präziseren Kern.
Antworte NUR mit der neuen Suchanfrage, ohne Erklärung.

Frage des Nutzers: {question}

Bereits erfolglos versucht:
{history}
""")


def reformulate(state: GraphState) -> dict:
    """Erzeugt eine neue Suchanfrage – mit Blick auf alle bisherigen Versuche."""
    print("--- REFORMULATE ---")

    history = "\n".join(f"- {q}" for q in state.get("query_history", []))

    chain    = REFORMULATE_PROMPT | llm
    response = chain.invoke({"question": state["question"], "history": history})
    new_query = response.content.strip()

    retry_count = state.get("retry_count", 0) + 1
    print(f"    → Neue Suchanfrage: '{new_query}' (Versuch {retry_count})")

    return {
        "search_query":  new_query,
        "retry_count":   retry_count,
        "query_history": [new_query],   # ← der Reducer hängt an, kein append()
    }

## Bedingte Kante – das Herzstück

`check_relevance` ist keine Node, sondern eine **Routing-Funktion**: sie verändert den
State nicht, sondern gibt nur einen String zurück, der bestimmt, welche Kante als nächstes
genommen wird. Genau das kann eine lineare Chain nicht.

In [ ]:
# --- CONDITIONAL EDGE ---

def check_relevance(state: GraphState) -> str:
    """Entscheidet, ob der Kontext relevant genug ist oder reformuliert werden muss."""
    best_score  = max(state["rerank_scores"]) if state["rerank_scores"] else 0.0
    retry_count = state.get("retry_count", 0)

    print("--- CHECK RELEVANCE ---")
    print(f"    Bester Score: {best_score:.3f} (Schwelle: {RELEVANCE_THRESHOLD})")
    print(f"    Bisherige Versuche: {retry_count} / {MAX_RETRIES}")

    if best_score >= RELEVANCE_THRESHOLD:
        print("    → RELEVANT – weiter zu Generate")
        return "relevant"

    if retry_count < MAX_RETRIES:
        print("    → NICHT RELEVANT – Suchanfrage wird reformuliert")
        return "not_relevant"

    print("    → NICHT RELEVANT, aber max. Versuche erreicht – Antwort mit letztem Ergebnis")
    return "relevant"   # Fallback: lieber eine schwache Antwort als gar keine

## Graph zusammenbauen

In [ ]:
# --- GRAPH ZUSAMMENBAUEN ---

workflow = StateGraph(GraphState)

# Knoten registrieren
workflow.add_node("retrieve_node",    retrieve)
workflow.add_node("rerank_node",      rerank)
workflow.add_node("generate_node",    generate)
workflow.add_node("reformulate_node", reformulate)

# Kanten definieren
workflow.add_edge(START, "retrieve_node")
workflow.add_edge("retrieve_node", "rerank_node")

# ⭐ Die bedingte Kante: check_relevance entscheidet den Weg
workflow.add_conditional_edges(
    "rerank_node",           # Nach diesem Knoten ...
    check_relevance,         # ... wird diese Funktion aufgerufen ...
    {                        # ... und ihr Rückgabewert bestimmt den nächsten Knoten:
        "relevant":     "generate_node",
        "not_relevant": "reformulate_node",
    },
)

# Der Zyklus: reformulate → retrieve (und von dort wieder rerank → check)
workflow.add_edge("reformulate_node", "retrieve_node")
workflow.add_edge("generate_node", END)

# Graph kompilieren
app = workflow.compile()

## Graph visualisieren

Vergleiche das Diagramm mit dem aus Notebook 2 – der Unterschied ist der ganze Punkt
dieses Notebooks.

In [ ]:
def display_graph(graph_app):
    """Zeigt den LangGraph als Mermaid-Diagramm an und speichert die Syntax."""
    mermaid_code = graph_app.get_graph().draw_mermaid()

    # Diagramm im Notebook rendern (via mermaid.ink)
    encoded = base64.b64encode(mermaid_code.encode()).decode()
    display.display(display.Image(url=f"https://mermaid.ink/img/{encoded}"))

    # Mermaid-Syntax als Datei speichern (optional)
    with open("rag_graph.mmd", "w", encoding="utf-8") as f:
        f.write(mermaid_code)


display_graph(app)

## Gradio-Interface

Die Quellenangaben sind hier klickbar: Gradio darf über `allowed_paths` auf den
Dokumenten-Ordner zugreifen, der Link springt per `#page=` direkt auf die Seite.
Gespeichert wird nur der Dateiname – liegt ein Dokument in einem Unterordner, fällt der
Eintrag auf reinen Text zurück.

In [ ]:
# Absoluter Pfad zum Dokumenten-Ordner – aus der Konfiguration, nicht hartkodiert
DOCUMENTS_ABS = os.path.abspath(DOC_SOURCE_DIR)


def chat_interface(question: str):
    """Verarbeitet eine Frage über den RAG-Graphen mit Hybrid Search."""
    yield "⏳ Hybrid Search, Reranking und Antwortgenerierung laufen..."

    result = app.invoke({
        "question":      question,
        "search_query":  question,
        "retry_count":   0,
        "query_history": [question],
    })

    answer = result["answer"]

    # Reranking-Info mit klickbaren Dokument-Links
    scores = result.get("rerank_scores", [])
    rerank_info = "\n\n---\n🎯 **Reranking-Scores (Top-Chunks):**\n"
    for meta, score in zip(result["metadata"], scores):
        source    = meta["source"]
        page      = meta["page"]
        file_path = os.path.join(DOCUMENTS_ABS, source)

        if os.path.exists(file_path):
            link = f"/gradio_api/file={quote(file_path)}#page={page}"
            rerank_info += f"- [{meta['id']}] [{source} (S. {page})]({link}): {score:.3f}\n"
        else:
            # z.B. wenn die Quelle in einem Unterordner liegt – dann kein Link
            rerank_info += f"- [{meta['id']}] {source} (S. {page}): {score:.3f}\n"

    # Hybrid-Search-Info
    hybrid_info = (
        f"\n🔀 **Hybrid Search:**\n"
        f"- Gewichtung: Vektor {ENSEMBLE_WEIGHTS[0]} / BM25 {ENSEMBLE_WEIGHTS[1]}\n"
        f"- Kandidaten je Retriever: {RETRIEVER_K}\n"
        f"- Fusion: Reciprocal Rank Fusion\n"
    )

    # Token-Statistik
    usage = result.get("token_usage", {})
    token_info = (
        f"\n📊 **Token-Statistik:**\n"
        f"- Input: {usage.get('prompt_tokens', 'N/A')}\n"
        f"- Output: {usage.get('completion_tokens', 'N/A')}\n"
        f"- Gesamt: {usage.get('total_tokens', 'N/A')}"
    )

    # Query-Historie
    retries = result.get("retry_count", 0)
    retry_info = ""
    if retries > 0:
        retry_info = f"\n\n🔄 Suchanfrage wurde {retries}× reformuliert:\n"
        for i, q in enumerate(result.get("query_history", [])):
            label = "Original" if i == 0 else f"Versuch {i}"
            retry_info += f"- **{label}:** {q}\n"

    yield answer + rerank_info + hybrid_info + token_info + retry_info


demo = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs=gr.Markdown(),
    title="RAG mit Hybrid Search, Reranking & Relevanz-Gate",
    description="Fragen an deine Dokumente – Vektor- und BM25-Suche kombiniert, mit Cross-Encoder-Reranking.",
    flagging_mode="never",
)

demo.launch(allowed_paths=[DOCUMENTS_ABS])

## Zum Nachdenken

- Stelle `ENSEMBLE_WEIGHTS` auf `[1.0, 0.0]` und dann auf `[0.0, 1.0]`. Bei welchen Fragen
  gewinnt welcher Retriever? Formuliere je eine Frage, bei der der eine klar besser ist.
- Warum arbeitet RRF mit Rängen statt mit den Original-Scores der Retriever?
- Wie viele Kandidaten bewertet der Cross-Encoder pro Frage im schlechtesten Fall –
  und was kostet das an Zeit?
- Der Graph fragt beide Retriever bei jedem Zyklus-Durchlauf erneut. Was ließe sich
  zwischenspeichern?